# Bet Simulator
In questo notebook è contenuto il bet silumator del progetto. L'obiettivo è comprendere, utilizzando i vari modelli e impostando una metodologia di scommesse, quanto sarebbe il guadagno ottenuto da ciascun modello (compresa la baseline casuale).

## Metodologia di scommessa
L'approcio che utilizziamo è semplice, scommettere 10 euro su ogni partita, sulla classe consigliata dal nostro modello,

## Modelli

Adesso verifichiamo qual è il guadagno dei singoli modelli

## Importazione dei dataset e merge
Per prima cosa facciamo la merge tra i due dataset (scommesse e partite) importati, di modo da avere tutti i dati necessari.  

In [4]:
import pandas as pd
import difflib
import numpy as np

class Odds:

    # Definizione delle variabili private di classe
    __odds_dataset: pd.DataFrame
    __match_dataset: pd.DataFrame
    __merged_dataset: pd.DataFrame

    def __init__(
            self,
            odds_dataframe_path: str,
            match_dataframe_path: str
    ) -> None:

        # Importazione e rimozione delle rows vuote 
        self.__odds_dataset = pd.read_csv(odds_dataframe_path)
        self.__match_dataset = pd.read_csv(match_dataframe_path)
        self.__odds_dataset = self.__odds_dataset.dropna()
        
        # Inizializzazione della variabile privata per il dataset finale
        self.__merged_dataset = pd.DataFrame()

    def merge_datasets(self) -> None:
        """
        Esegue l'allineamento e l'affiancamento dei dataset in base alla data 
        e al nome delle squadre (tramite fuzzy matching).
        Salva il risultato nella variabile privata __merged_dataset.
        """
        
        # 1. Normalizzazione delle date
        self.__match_dataset['Match_Date_Temp'] = pd.to_datetime(self.__match_dataset['Date']).dt.date
        self.__odds_dataset['Odds_Date_Temp'] = pd.to_datetime(self.__odds_dataset['matchDate'], dayfirst=True).dt.date

        # 2. Risoluzione dei nomi delle squadre (Fuzzy Matching)
        match_teams = pd.concat([self.__match_dataset['HomeTeam'], self.__match_dataset['AwayTeam']]).unique()
        odds_teams = pd.concat([self.__odds_dataset['homeTeam'], self.__odds_dataset['awayTeam']]).unique()

        team_mapping = {}
        for team in odds_teams:
            matches = difflib.get_close_matches(team, match_teams, n=1, cutoff=0.5)
            team_mapping[team] = matches[0] if matches else team

        self.__odds_dataset['homeTeam_Norm'] = self.__odds_dataset['homeTeam'].map(team_mapping)
        self.__odds_dataset['awayTeam_Norm'] = self.__odds_dataset['awayTeam'].map(team_mapping)

        # 3. Affiancamento dei due dataset
        self.__merged_dataset = pd.merge(
            left=self.__match_dataset,
            right=self.__odds_dataset,
            left_on=['Match_Date_Temp', 'HomeTeam', 'AwayTeam'],
            right_on=['Odds_Date_Temp', 'homeTeam_Norm', 'awayTeam_Norm'],
            how='inner' 
        )

        # 4. Pulizia delle colonne temporanee create per il merge
        col_rimuovere_match = ['Match_Date_Temp']
        col_rimuovere_odds = ['Odds_Date_Temp', 'homeTeam_Norm', 'awayTeam_Norm']
        
        # Le puliamo sia dai dataset originali che dal dataset finale
        self.__match_dataset.drop(columns=col_rimuovere_match, inplace=True, errors='ignore')
        self.__odds_dataset.drop(columns=col_rimuovere_odds, inplace=True, errors='ignore')
        self.__merged_dataset.drop(columns=col_rimuovere_match + col_rimuovere_odds, inplace=True, errors='ignore')

        print(f"Merge completato! Numero di righe nel dataset finale: {len(self.__merged_dataset)}")

    def get_merged_dataset(self) -> pd.DataFrame:
        """
        Restituisce il dataset unito in modo sicuro.
        """
        if self.__merged_dataset.empty:
            print("Attenzione: il dataset unito è vuoto. Assicurati di aver eseguito merge_datasets().")
            
        return self.__merged_dataset
    
    def simulate_montecarlo(
        self, 
        n_simulations: int = 1000, 
        sample_fraction: float = 0.7, 
        initial_bankroll: float = 1000.0, 
        bet_size: float = 10.0
    ) -> dict:
        """
        Esegue una simulazione Monte Carlo su campioni casuali del dataset unito.
        Testa una strategia di scommessa per valutarne ROI e varianza.
        """
        if self.__merged_dataset.empty:
            print("Errore: Dataset vuoto. Esegui merge_datasets() prima della simulazione.")
            return {}

        # Assicuriamoci che le colonne necessarie esistano 
        # FTR = Full Time Result (H, D, A), H = Quote vittoria casa
        if 'FTR' not in self.__merged_dataset.columns or 'H' not in self.__merged_dataset.columns:
            print("Errore: Colonne 'FTR' (risultato) o 'H' (quota casa) non trovate nel dataset.")
            return {}

        final_bankrolls = []
        
        print(f"Avvio di {n_simulations} simulazioni Monte Carlo...")

        for _ in range(n_simulations):
            bankroll = initial_bankroll
            
            # 1. Peschiamo un campione casuale dal dataset (es. 70% delle partite)
            # Questo introduce la casualità tipica del Monte Carlo
            sample_df = self.__merged_dataset.sample(frac=sample_fraction, replace=True)

            # 2. Iteriamo sulle partite del campione per applicare la strategia
            # Qui usiamo itertuples() per le massime performance come discusso in precedenza
            for row in sample_df.itertuples():
                # ESEMPIO STRATEGIA: Scommettiamo sempre sulla squadra di casa (H)
                quota_casa = float(row.H)
                risultato_reale = row.FTR
                
                # Sottraiamo l'importo scommesso
                bankroll -= bet_size
                
                # Se la squadra di casa ha vinto, aggiungiamo la vincita
                if risultato_reale == 'H':
                    vincita = bet_size * quota_casa
                    bankroll += vincita
                    
                # Se il bankroll scende a 0, siamo in bancarotta
                if bankroll <= 0:
                    bankroll = 0
                    break
                    
            final_bankrolls.append(bankroll)

        # 3. Analisi dei risultati aggregati
        final_bankrolls = np.array(final_bankrolls)
        avg_bankroll = np.mean(final_bankrolls)
        profitto_medio = avg_bankroll - initial_bankroll
        roi_medio = (profitto_medio / initial_bankroll) * 100
        
        risultati = {
            "Simulazioni": n_simulations,
            "Bankroll_Iniziale": initial_bankroll,
            "Bankroll_Medio_Finale": round(avg_bankroll, 2),
            "Profitto_Medio": round(profitto_medio, 2),
            "ROI_Medio_%": round(roi_medio, 2),
            "Scenario_Peggiore": round(np.min(final_bankrolls), 2),
            "Scenario_Migliore": round(np.max(final_bankrolls), 2),
            "Rischio_Bancarotta_%": round((np.sum(final_bankrolls == 0) / n_simulations) * 100, 2)
        }

        return risultati

In [3]:
# Dopo aver instanziato la classe e fatto il merge:
gestore_quote = Odds("static/odds/odds.csv", "static/da-result/result.csv")
gestore_quote.merge_datasets()

# Lancia il Monte Carlo
risultati_mc = gestore_quote.simulate_montecarlo(
    n_simulations=1000, 
    initial_bankroll=500.0, 
    bet_size=10.0
)

# Stampa i risultati
for metrica, valore in risultati_mc.items():
    print(f"{metrica}: {valore}")

/tmp/ipykernel_54484/2182609087.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.__odds_dataset['Odds_Date_Temp'] = pd.to_datetime(self.__odds_dataset['matchDate'], dayfirst=True).dt.date


Merge completato! Numero di righe nel dataset finale: 734
Avvio di 1000 simulazioni Monte Carlo...


NameError: name 'np' is not defined